# 06 · The Wave Equation — A Hyperbolic PDE

**Level:** Intermediate
**Prerequisites:** `03_heat_equation`

The wave equation is second-order in *both* space and time. Unlike the diffusive
heat equation (which smooths everything out), waves **preserve** features and
oscillate — which makes them a stiffer test of a PINN's ability to represent
higher frequencies. Acoustics, seismology, electromagnetics and vibration all
live here.

### What you will learn
1. A PDE that is **second-order in time** → needs an initial *velocity* too
2. Enforcing `∂u/∂t(x,0)` with `autograd`
3. Seeing spectral bias bite harder than for diffusion

### References
- Raissi, Perdikaris & Karniadakis (2019)
- Wang, Yu & Perdikaris (2022), *When and why PINNs fail to train: A NTK perspective*, J. Comput. Phys.


## 1 · Problem statement

$$\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2},
\qquad x\in[0,1],\; t\in[0,1]$$

with a standing-wave setup:

$$u(x,0)=\sin(\pi x),\quad \frac{\partial u}{\partial t}(x,0)=0,\quad
u(0,t)=u(1,t)=0.$$

Exact solution: $u(x,t)=\sin(\pi x)\cos(\pi c\,t)$.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

c = 1.0
def exact(x, t):
    return np.sin(np.pi*x)*np.cos(np.pi*c*t)


In [ ]:
class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2: seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x, t):
        return self.net(torch.cat([x,t],1))

model = PINN([2,64,64,64,64,1]).to(device)


In [ ]:
N_f, N_ic, N_bc = 6000, 400, 300
x_f = torch.rand(N_f,1,device=device); t_f = torch.rand(N_f,1,device=device)
x_f.requires_grad_(True); t_f.requires_grad_(True)

x_ic = torch.rand(N_ic,1,device=device); t_ic = torch.zeros(N_ic,1,device=device)
t_ic.requires_grad_(True)                       # only t needs grad (for the velocity IC)
u_ic = torch.sin(np.pi*x_ic).detach()           # target: a constant, no graph

t_bc = torch.rand(N_bc,1,device=device)
x_b0 = torch.zeros(N_bc,1,device=device); x_b1 = torch.ones(N_bc,1,device=device)

def residual(x,t):
    u = model(x,t)
    u_t = torch.autograd.grad(u,t,torch.ones_like(u),create_graph=True)[0]
    u_tt= torch.autograd.grad(u_t,t,torch.ones_like(u_t),create_graph=True)[0]
    u_x = torch.autograd.grad(u,x,torch.ones_like(u),create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x,x,torch.ones_like(u_x),create_graph=True)[0]
    return u_tt - c**2*u_xx

def loss_fn():
    L_f = torch.mean(residual(x_f,t_f)**2)
    # IC on value
    u0 = model(x_ic,t_ic)
    L_u0 = torch.mean((u0 - u_ic)**2)
    # IC on velocity du/dt(x,0)=0
    u0_t = torch.autograd.grad(u0,t_ic,torch.ones_like(u0),create_graph=True)[0]
    L_v0 = torch.mean(u0_t**2)
    # BCs
    L_bc = torch.mean(model(x_b0,t_bc)**2) + torch.mean(model(x_b1,t_bc)**2)
    return L_f + 20*L_u0 + 20*L_v0 + 10*L_bc


In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 15000
for ep in range(EPOCHS):
    opt.zero_grad(); loss = loss_fn(); loss.backward(); opt.step()
    if ep % 3000 == 0: print(f"epoch {ep:5d} | loss {loss.item():.3e}")
# L-BFGS polish
opt2 = torch.optim.LBFGS(model.parameters(), max_iter=2000, line_search_fn="strong_wolfe")
def closure():
    opt2.zero_grad(); l = loss_fn(); l.backward(); return l
opt2.step(closure)
print("final loss:", loss_fn().item())


In [ ]:
nx, nt = 100, 100
xs = np.linspace(0,1,nx); ts = np.linspace(0,1,nt)
X,Tg = np.meshgrid(xs,ts)
xf = torch.tensor(X.ravel(),dtype=torch.float32,device=device).view(-1,1)
tf = torch.tensor(Tg.ravel(),dtype=torch.float32,device=device).view(-1,1)
with torch.no_grad():
    U = model(xf,tf).cpu().numpy().reshape(nt,nx)
Ue = exact(X,Tg)
print(f"relative L2 error: {np.linalg.norm(U-Ue)/np.linalg.norm(Ue):.3e}")

fig,ax = plt.subplots(1,3,figsize=(15,4))
for a,(dat,ttl) in zip(ax,[(U,'PINN'),(Ue,'exact'),(np.abs(U-Ue),'|error|')]):
    im=a.pcolormesh(X,Tg,dat,shading='auto'); a.set_xlabel('x'); a.set_ylabel('t')
    a.set_title(ttl); fig.colorbar(im,ax=a)
plt.tight_layout(); plt.show()


## 2 · Key difference from the heat equation

Because the wave equation is **second-order in time**, one initial condition is
not enough — the solution's future depends on both the initial displacement
`u(x,0)` and the initial velocity `∂u/∂t(x,0)`. We enforce the velocity condition
with a nested `autograd` call, exactly as we did for the initial velocity of the
oscillator in notebook 02.

Waves are also *non-dissipative*: errors do not decay, so a PINN that is slightly
off early stays off. This is why the wave equation is a favourite stress test in
papers analysing PINN training failure (Wang, Yu & Perdikaris 2022).

## 3 · Exercises
1. Increase the wave number: `u(x,0)=sin(4πx)`. Watch the error climb — pure
   spectral bias. Add Fourier features (Advanced track) to fix it.
2. Add a moving source term and compare with a finite-difference reference.
3. Try `c=2` and verify the exact solution changes accordingly.

**Next (Advanced track):** `07_inverse_parameter_estimation` — using PINNs to
*discover* unknown physical parameters from sparse data.
